# Salvar e carregar os modelos de dificuldade lexical

Este notebook adapta o exemplo fornecido pela professora para salvar o SVM, modelo com melhor Macro-F1 na avaliação final, e as alternativas Regressão Logística e LDA.

## Ambiente

Execute a partir da pasta `ml/` do projeto, com as dependências instaladas. No Google Colab, envie a pasta `ml/` ou monte o Drive e ajuste os caminhos abaixo.

In [1]:
# Opcional no Google Colab:
# from google.colab import drive
# drive.mount('/content/drive')

import sys
from pathlib import Path
import joblib

# Permite importar alfabetiza_ml sem instalar o pacote (executar a partir da pasta ml/)
sys.path.insert(0, str(Path.cwd() / 'src'))

from alfabetiza_ml.train_and_save_human_models import train_and_save, training_data


## Treinar e salvar

Os três modelos são treinados com 339 palavras humanas consolidadas: 94 fáceis, 141 médias e 104 difíceis.

In [2]:
artifacts = train_and_save()
artifacts


{'manifest': '/Users/torquato/Documents/GitHub/DSM-P5-G01-2026-02/ml/artifacts/modelos_dificuldade_humana_manifest.json',
 'models': {'SVM': '/Users/torquato/Documents/GitHub/DSM-P5-G01-2026-02/ml/artifacts/svm_dificuldade_humana.joblib',
  'LR': '/Users/torquato/Documents/GitHub/DSM-P5-G01-2026-02/ml/artifacts/lr_dificuldade_humana.joblib',
  'LDA': '/Users/torquato/Documents/GitHub/DSM-P5-G01-2026-02/ml/artifacts/lda_dificuldade_humana.joblib'},
 'source_corpus': '/Users/torquato/Documents/GitHub/DSM-P5-G01-2026-02/ml/data/Corpus_SANTOS_TONI_FDC_tratado_sem_repeticoes_final.xlsx',
 'training_rows': 339,
 'class_counts': {'medio': 141, 'dificil': 104, 'facil': 94},
 'feature_names': ['syllable_count',
  'complex_syllable_count',
  'cluster_count',
  'max_shape_length',
  'coda_count',
  'glide_count',
  'word_length',
  'accented',
  'digraph_count',
  'complex_grapheme_count',
  'log_frequency'],
 'random_state': 7}

## Carregar o SVM

Equivale ao `joblib.load(...)` do exemplo da professora. Este foi o melhor modelo na avaliação final.

In [3]:
svm_path = Path(artifacts['models']['SVM'])
loaded_svm = joblib.load(svm_path)
svm_model = loaded_svm['model']

X, y, metadata = training_data()
predicoes_svm = svm_model.predict(X)

print('Modelo:', type(svm_model).__name__)
print('Linhas usadas no treino:', metadata['training_rows'])
print('Primeiras previsões:', predicoes_svm[:10])


Modelo: Pipeline
Linhas usadas no treino: 339
Primeiras previsões: ['facil' 'dificil' 'medio' 'dificil' 'medio' 'dificil' 'dificil' 'medio'
 'facil' 'facil']


## Usar o modelo vencedor em palavras do corpus

Carrega o SVM salvo e classifica palavras do corpus tratado, sem treinar novamente.

In [4]:
import pandas as pd
from alfabetiza_ml.benchmark_human_ratings import CORPUS, features

corpus = pd.read_excel(CORPUS).drop_duplicates('PALAVRA')
exemplos = corpus[corpus['PALAVRA'].isin(['casa', 'bola', 'prato', 'abacaxi', 'vermelho', 'brincar'])].copy()
exemplos['PREVISAO_SVM'] = svm_model.predict(features(exemplos))
exemplos[['PALAVRA', 'ESTSILABICA', 'PREVISAO_SVM']]


,PALAVRA,ESTSILABICA,PREVISAO_SVM
2,abacaxi,VCCV,dificil
971,bola,&CV-CV%,facil
1076,brincar,&CCV-CVC%,medio
1353,casa,&CV-CV%,facil
5816,prato,&CCV-CV%,medio
7450,vermelho,&CVC-CV-CV%,dificil


## Carregar a Regressão Logística e o LDA

A Regressão Logística e o LDA foram salvos como alternativas para comparação e reuso.

In [5]:
for model_name in ('LR', 'LDA'):
    loaded = joblib.load(Path(artifacts['models'][model_name]))
    print(model_name, type(loaded['model']).__name__, loaded['model'].predict(X)[:10])


LR Pipeline ['facil' 'dificil' 'dificil' 'dificil' 'dificil' 'dificil' 'dificil'
 'medio' 'facil' 'facil']
LDA Pipeline ['facil' 'dificil' 'dificil' 'dificil' 'medio' 'medio' 'dificil' 'medio'
 'facil' 'facil']


## Arquivos gerados

- `ml/artifacts/svm_dificuldade_humana.joblib` — modelo vencedor salvo.
- `ml/artifacts/lr_dificuldade_humana.joblib` e `lda_dificuldade_humana.joblib` — modelos alternativos salvos.
- `ml/artifacts/modelos_dificuldade_humana_manifest.json` — metadados, atributos e origem da base de treinamento.